In [6]:
from pathlib import Path
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Maciej\PycharmProjects\CyberLab")
os.chdir(PROJECT_ROOT)

def run_module(module: str, *arguments: object) -> None:
    command = [
        sys.executable,
        "-u",
        "-m",
        module,
        *[str(argument) for argument in arguments],
    ]

    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(
            return_code,
            command,
        )

# Przygotowanie danych standardowych

In [7]:
run_module(
    "src.preprocess",
    "--mode", "standard",
)


PREPROCESSING CICIDS2017
Tryb: standard
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Monday-WorkingHours.pcap_ISCX.csv
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Tuesday-WorkingHours.pcap_ISCX.csv
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Wednesday-workingHours.pcap_ISCX.csv
Total records after merge: 2830743


# Klasyfikacja binarna

In [ ]:
run_module(
    "scripts.train",
    "--processed-path", "data/processed",
    "--target", "Label_binary",
    "--models", "all",
    "--best-params-path", "results/tuning_binary/best_params_all.json",
    "--results-path", "results/final_binary",
    "--models-path", "models/final_binary",
)


# Klasyfikacja wieloklasowa

In [ ]:
run_module(
    "scripts.train",
    "--processed-path", "data/processed",
    "--target", "AttackClass",
    "--models", "all",
    "--best-params-path", "results/tuning_multiclass/best_params_all.json",
    "--results-path", "results/final_multiclass",
    "--models-path", "models/final_multiclass",
)


# Przygotowanie leave-one-attack-class-out

In [8]:
run_module(
    "src.preprocess",
    "--mode", "leave-one-out",
    "--excluded-attacks", "all",
)

PREPROCESSING CICIDS2017
Tryb: leave-one-out
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Monday-WorkingHours.pcap_ISCX.csv
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Tuesday-WorkingHours.pcap_ISCX.csv
Loading: C:\Users\Maciej\PycharmProjects\CyberLab\data\raw\CICIDS2017\Wednesday-workingHours.pcap_ISCX.csv
Total records after merge: 283

KeyboardInterrupt: 

# Klasyfikacja Leave-one-out


In [ ]:
run_module(
    "scripts.run_experiments",
    "--experiment", "leave-one-out",
    "--target", "Label_binary",
    "--models", "all",
    "--best-params-path", "results/tuning_binary/best_params_all.json",
    "--continue-on-error",
)


# Przygotowanie międzydniowego

In [ ]:
TRAIN_FILES = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv"
]

TEST_FILES = [
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
]

In [ ]:
run_module(
    "src.preprocess",
    "--mode", "cross-day",
    "--scenario-name", "train_selected_days_test_selected_day",
    "--train-files", *TRAIN_FILES,
    "--test-files", *TEST_FILES,
)


# Klasyfikacja międzydniowa

In [ ]:
run_module(
    "scripts.run_experiments",
    "--experiment", "cross-day",
    "--target", "Label_binary",
    "--models", "all",
    "--best-params-path", "results/tuning_binary/best_params_all.json",
    "--continue-on-error",
)

# Wczytanie wyników standardowych

In [ ]:
binary_path = Path(
    "results/final_binary/model_comparison_Label_binary.csv"
)
multiclass_path = Path(
    "results/final_multiclass/model_comparison_AttackClass.csv"
)

binary_results = (
    pd.read_csv(binary_path)
    if binary_path.exists()
    else pd.DataFrame()
)

multiclass_results = (
    pd.read_csv(multiclass_path)
    if multiclass_path.exists()
    else pd.DataFrame()
)

display(binary_results)
display(multiclass_results)


# Porównanie Macro F1 modeli

In [ ]:
def plot_model_metric(
    frame: pd.DataFrame,
    metric: str,
    title: str,
) -> None:
    if frame.empty:
        print("Brak danych.")
        return

    required = {"model", metric}
    if not required.issubset(frame.columns):
        print(f"Brak kolumn: {required - set(frame.columns)}")
        return

    plot_data = frame[["model", metric]].sort_values(
        metric,
        ascending=False,
    )

    ax = plot_data.plot(
        x="model",
        y=metric,
        kind="bar",
        legend=False,
        figsize=(10, 5),
    )
    ax.set_xlabel("Model")
    ax.set_ylabel(metric)
    ax.set_title(title)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


plot_model_metric(
    binary_results,
    "macro_f1",
    "Macro F1 — klasyfikacja binarna",
)

plot_model_metric(
    multiclass_results,
    "macro_f1",
    "Macro F1 — klasyfikacja wieloklasowa",
)


# Macierz pomyłek z pliku CSV

In [ ]:
def plot_confusion_matrix_csv(
    csv_path: str | Path,
    title: str,
    normalize: bool = False,
) -> None:
    csv_path = Path(csv_path)

    if not csv_path.exists():
        print(f"Nie istnieje: {csv_path}")
        return

    matrix = pd.read_csv(
        csv_path,
        index_col=0,
    )

    values = matrix.to_numpy(dtype=float)

    if normalize:
        row_sums = values.sum(axis=1, keepdims=True)
        values = np.divide(
            values,
            row_sums,
            out=np.zeros_like(values),
            where=row_sums != 0,
        )

    fig, ax = plt.subplots(figsize=(9, 7))
    image = ax.imshow(values)

    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels(
        matrix.columns,
        rotation=45,
        ha="right",
    )

    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)

    ax.set_xlabel("Klasa przewidziana")
    ax.set_ylabel("Klasa rzeczywista")
    ax.set_title(title)

    fig.colorbar(image, ax=ax)
    plt.tight_layout()
    plt.show()

#example
plot_confusion_matrix_csv(
    "results/final_multiclass/confusion_matrix_random_forest_AttackClass.csv",
    "Macierz pomyłek — Random Forest",
    normalize=True,
)


# Wyniki leave-one-out

In [ ]:
leave_one_out_path = Path(
    "results/experiments/leave_one_out/"
    "leave_one_out_summary_Label_binary.csv"
)

leave_one_out_results = (
    pd.read_csv(leave_one_out_path)
    if leave_one_out_path.exists()
    else pd.DataFrame()
)

display(leave_one_out_results)


# Heatmapa wykrywania niewidzianych ataków

In [ ]:
def plot_unseen_attack_heatmap(
    frame: pd.DataFrame,
) -> None:
    required = {
        "held_out_attack",
        "model",
        "unseen_attack_detection_rate",
    }

    if frame.empty:
        print("Brak wyników leave-one-out.")
        return

    if not required.issubset(frame.columns):
        print(f"Brak kolumn: {required - set(frame.columns)}")
        return

    pivot = frame.pivot_table(
        index="held_out_attack",
        columns="model",
        values="unseen_attack_detection_rate",
        aggfunc="mean",
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    image = ax.imshow(pivot.to_numpy(dtype=float))

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(
        pivot.columns,
        rotation=30,
        ha="right",
    )

    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    ax.set_xlabel("Model")
    ax.set_ylabel("Niewidziany typ ataku")
    ax.set_title("Współczynnik wykrywania niewidzianych ataków")

    fig.colorbar(image, ax=ax)
    plt.tight_layout()
    plt.show()


plot_unseen_attack_heatmap(
    leave_one_out_results
)


# Wyniki eksperymentu międzydniowego

In [ ]:
cross_day_path = Path(
    "results/experiments/cross_day/"
    "cross_day_summary_Label_binary.csv"
)

cross_day_results = (
    pd.read_csv(cross_day_path)
    if cross_day_path.exists()
    else pd.DataFrame()
)

display(cross_day_results)


# Porównanie standard vs cross-day

In [ ]:
def compare_standard_and_cross_day(
    standard: pd.DataFrame,
    cross_day: pd.DataFrame,
    metric: str = "macro_f1",
) -> pd.DataFrame:
    if standard.empty or cross_day.empty:
        return pd.DataFrame()

    standard_part = standard[
        ["model", metric]
    ].rename(
        columns={metric: f"{metric}_standard"}
    )

    cross_day_part = (
        cross_day.groupby("model", as_index=False)[metric]
        .mean()
        .rename(columns={metric: f"{metric}_cross_day"})
    )

    comparison = standard_part.merge(
        cross_day_part,
        on="model",
        how="inner",
    )

    comparison[f"{metric}_drop"] = (
        comparison[f"{metric}_standard"]
        - comparison[f"{metric}_cross_day"]
    )

    return comparison.sort_values(
        f"{metric}_drop",
        ascending=True,
    )


cross_day_comparison = compare_standard_and_cross_day(
    binary_results,
    cross_day_results,
    metric="macro_f1",
)

display(cross_day_comparison)


# Benchmark latency i throughput

In [ ]:
benchmark_binary_path = Path(
    "results/final_binary/realtime_benchmark_Label_binary.csv"
)

benchmark_binary = (
    pd.read_csv(benchmark_binary_path)
    if benchmark_binary_path.exists()
    else pd.DataFrame()
)

display(benchmark_binary)


In [ ]:
def plot_benchmark(
    frame: pd.DataFrame,
    metric: str,
    title: str,
) -> None:
    required = {"model", "batch_size", metric}

    if frame.empty:
        print("Brak wyników benchmarku.")
        return

    if not required.issubset(frame.columns):
        print(f"Brak kolumn: {required - set(frame.columns)}")
        return

    fig, ax = plt.subplots(figsize=(10, 6))

    for model_name, model_frame in frame.groupby("model"):
        ordered = model_frame.sort_values("batch_size")
        ax.plot(
            ordered["batch_size"],
            ordered[metric],
            marker="o",
            label=model_name,
        )

    ax.set_xlabel("Batch size")
    ax.set_ylabel(metric)
    ax.set_title(title)
    ax.legend()
    ax.set_xscale("log", base=2)
    plt.tight_layout()
    plt.show()


plot_benchmark(
    benchmark_binary,
    "throughput_flows_per_second",
    "Throughput modeli — klasyfikacja binarna",
)

plot_benchmark(
    benchmark_binary,
    "p95_batch_latency_ms",
    "P95 latency — klasyfikacja binarna",
)
